# New run: push a config.json

Builds a `RunConfig` (validated by `run_config.RunConfig`) for one run and pushes it to `runs/{run_id}/config.json` on the `trainvols` volume -- the notebook equivalent of `push_config.py`.

Example below: two jobs, `ETL` and `Count`, where `Count` depends on `ETL` finishing first. job_uid IS the class name in `jobs.py` -- `RunConfig` checks that at construction, so a typo or a renamed class fails right here, not when `main.py` tries to launch it.

In [1]:
import io
import time

import modal

from config import VOLUME_NAME, get_git_commit
from jobs import ETL, artifact_name, preflight_check
from run_config import JobEntry, ResourcesSpec, RunConfig
from main import volume

## Build and validate the config

Edit `run_id` and the job entries below. Construction itself is the validation -- a bad type, shape, or job_uid raises right here, not three cells later.

In [2]:
run_id = "count-etl-demo"  # change per run

config = RunConfig(
    metadata={"run_id": run_id, "pushed_ts": time.time()},
    jobs={
        "ETL": JobEntry(
            resources=ResourcesSpec(cpu=2),
        ),
        "Count": JobEntry(
            # a path to ETL's actual artifact file, not ETL's name --
            # ETL.job_uid is what ETL.start() itself names the file.
            dependencies=[f"runs/{run_id}/{artifact_name(ETL.job_uid)}"],
        ),
    },
)
config

RunConfig(metadata={'run_id': 'count-etl-demo', 'pushed_ts': 1787157052.3245928}, jobs={'ETL': JobEntry(parameters={}, dependencies=[], resources=ResourcesSpec(cpu=2.0, gpu_type=None, gpu_count=None)), 'Count': JobEntry(parameters={}, dependencies=['runs/count-etl-demo/etl_artifact.txt'], resources=ResourcesSpec(cpu=None, gpu_type=None, gpu_count=None))})

## A richer example: dependencies that aren't job names

`Source` downloads a couple of texts. `Tokenize` fits on a chosen subset
of them (`fit_source`) plus its own hyperparameters -- together, its whole
identity -- folded into `tok_hash`, a short content hash. Its output lives
at a path *named by that hash*, not by "the Tokenize job," so two runs
that ask for the identical tokenizer configuration land on the same
already-fitted output instead of each fitting their own. `Train` depends
on that exact path: not on `Tokenize` in general, on the one configuration
its own `tokenizer_hash` parameter names.

In [3]:
NAME = "run name"  # the readable half of the run_id
VOCAB_SIZE = 2000  # sizes the embedding table *and* fits the encoder


TRAIN_SET = ["odyssey","mobydick"],
VALID_SET = ["montecristo", "romeojuliet"]
TOKEN_SET = ["odyssey"]

TOKENIZER = {
    "kind": "bpe",
    "params": {
        "vocab_size": VOCAB_SIZE,
        "special_tokens": ["<|endoftext|>", "<|begin|>", "<|end|>"],
    },
    "fit_sources": TOKEN_SET,
}

TAGS = (
    [f"run:{NAME}"]
    + [f"train:{uid}" for uid in TRAIN_SET]
    + [f"valid:{uid}" for uid in VALID_SET]
    + [f"enc:{tokenizer_uid}"]
)


PRETRAINING = {
    "description": NAME,
    "seed": 0,
    "model_class": "TransformerLM",
    "model_params": {
        "vocab_size": VOCAB_SIZE,
        "context_length": 64,
        "num_layers": 1,
        "d_model": 64,
        "d_ff": 128,
        "num_heads": 1,
        "rope_theta": 10000,
        "device": "cuda",  # has to agree with app.GPU; the preflight checks it
        "dtype": None,
    },
    "optimizer_class": "torch.optim.AdamW",
    "optimizer_params": {
        "lr": 0.001,
        "betas": (0.9, 0.999),
        "weight_decay": 0.1,
        "eps": 1e-8,
    },
    "lr_schedule_fn": "lr_cosine_schedule",
    "lr_schedule_params": {
        "max_learning_rate": 0.001,
        "min_learning_rate": 0.0001,
        "warmup_iters": 30,
        "cosine_cycle_iters": 1000,
    },
    # No train_path/valid_path: a run trains on its own folder's train.bin and
    # valid.bin, which the ETL gathers from the sources below. Naming paths here
    # would be a second way of saying what `sources` already says.
    "training": {
        "total_steps": 5000,
        "batch_size": 32,
        "val_every": 10,
        "save_every": 1000,
        "gpu_check_every": 50,
        "max_norm": 1.0,  # gradient-clipping threshold, passed to clip_grad_norm_
    },
    "sources": TRAIN_SET + VALID_SET,
    "tokenizer": {tokenizer_uid},
    # Passed straight to wandb.init: project/name/notes/tags become native W&B
    # fields, anything else here becomes a flat config column. Delete this whole
    # block to turn W&B off -- WandbRun keys off its presence.
    #
    # Tags are set once, at run creation, and are sticky: on every resume wandb
    # re-attaches to the existing run and the server's tags overwrite whatever init
    # passes. Fine here -- config.json is write-once, so the sources cannot change
    # mid-run -- but editing TAGS and resuming changes nothing in W&B.
   
}

METADATA = {
        "git_commit": get_git_commit(True),
        "project": "llm-pretraining",
        "name": NAME,
        "notes": "",
        "tags": TAGS,
    },

NameError: name 'tokenizer_uid' is not defined

In [ ]:
import hashlib
import json


def tokenizer_hash(config: dict) -> str:
    """A short, deterministic id for a tokenizer's configuration -- same
    config, same hash, so a fitted tokenizer is found and reused across
    runs instead of re-fit every time. In practice this belongs in a
    shared module the Tokenize job itself imports too -- the notebook and
    the job must compute it exactly the same way, or a dependency path
    built here points at a tokenizer the job would never produce.
    """
    blob = json.dumps(config, sort_keys=True).encode()
    return f"{config["algorithm"]}-{config["vocab_size"]}-{hashlib.sha256(blob).hexdigest()[:12]}"


run_id = "transformer"  # change per run

# A couple of sources -- each downloaded to data/sources/{name}/content.txt,
# shared and not run-scoped, so a later run naming "odyssey" again reuses
# this download instead of refetching it.
sources = {
    "odyssey": {
        "urls": ["https://gutenberg.org/cache/epub/1727/pg1727.txt"],
        "tags": ["source:gutenberg", "lang:en", "genre:fiction", "author:homer"],
    },
    "mobydick": {
        "urls": ["https://www.gutenberg.org/cache/epub/2701/pg2701.txt"],
        "tags": ["source:gutenberg", "lang:en", "genre:fiction", "author:melville"],
    },
    "montecristo": {
        urls: [https://gutenberg.org/cache/epub/1184/pg1184.txt],
        tags: [source:gutenberg, lang:en, genre:fiction, author:dumas]
    },
    "romeojuliet": {urls: [https://gutenberg.org/cache/epub/1513/pg1513.txt],
                tags: [source:gutenberg, lang:en, genre:drama, author:shakespeare]},
}

TRAIN_SOURCE = ["odyssey","mobydick"]
VALID_SOURCE = ["romeojuliet","montecristo"]
FIT_SOURCE = ["odyssey"]

# Everything that makes this tokenizer a distinct tokenizer: which
# source(s) it's fit on, plus its own hyperparameters.
tokenizer_config = {
    "fit_source": FIT_SOURCE,
    "vocab_size": 8000,
    "algorithm": "bpe",
}
tok_hash = tokenizer_hash(tokenizer_config)

config = RunConfig(
    metadata={"run_id": run_id, "pushed_ts": time.time()},
    jobs={
        "Source": JobEntry(
            parameters={"sources": sources},
        ),
        # Depends on every fit_source's downloaded content -- not on the
        # Source job_uid, on the specific files it needs.
        "Tokenize": JobEntry(
            parameters=tokenizer_config,
            dependencies=[
                f"data/sources/{name}/content.txt" for name in tokenizer_config["fit_source"]
            ],
        ),
        # Depends on the one tokenizer whose hash matches this exact
        # choice of algorithm/vocab_size/fit_source -- not on "the
        # Tokenize job" in general, on that configuration's output.
        "Train": JobEntry(
            parameters={"tokenizer": tok_hash, 
                        "seed": 42
                        },
            dependencies=[f"data/tokenizers/{tok_hash}/tokenizer.json",
                            f"data/tokenizers/{tok_hash}/bin/mobydick.bin",
                            f"data/tokenizers/{tok_hash}/bin/odyssey.bin",
                          ],
        ),
    },
)
config

In [ ]:
config

## Push it to the volume

Refuses to overwrite an existing `runs/{run_id}/config.json` unless `force = True` -- same rule `push_config.py` follows.

In [ ]:
force = True

volume = modal.Volume.from_name(VOLUME_NAME, create_if_missing=True)
remote_path = f"runs/{run_id}/config.json"
payload = config.model_dump_json(indent=2).encode()

try:
    with volume.batch_upload(force=force) as batch:
        batch.put_file(io.BytesIO(payload), remote_path)
    print(f"pushed config to {remote_path}")
except FileExistsError:
    print(f"{remote_path} already exists -- set force = True to overwrite")

## Launch it

```
uv run modal run main.py::launch_job --run-id count-etl-demo --job ETL
uv run modal run main.py::launch_job --run-id count-etl-demo --job Count
```

`Count` refuses to launch until `ETL` has left its artifact behind. No registration step needed elsewhere -- `main.py` resolves `job_uid` straight to the `jobs.py` class of that name.

In [ ]:
from jobs import Count
import logging
logger = logging.getLogger()

class Worker:
    def __init__(self):
        pass
    def confirm_lease():
        return True

In [ ]:
worker = Worker()
job = Count('runid',logger, worker)


sources:
  montecristo: {urls: [https://gutenberg.org/cache/epub/1184/pg1184.txt],
                tags: [source:gutenberg, lang:en, genre:fiction, author:dumas]}
  romeojuliet: {urls: [https://gutenberg.org/cache/epub/1513/pg1513.txt],
                tags: [source:gutenberg, lang:en, genre:drama, author:shakespeare]}
  odyssey:     {urls: [https://gutenberg.org/cache/epub/1727/pg1727.txt],
                tags: [source:gutenberg, lang:en, genre:fiction, author:homer]}
  mobydick:    {urls: [https://www.gutenberg.org/cache/epub/2701/pg2701.txt],
                tags: [source:gutenberg, lang:en, genre:fiction, author:melville]}

In [ ]:
JobEntry(
        parameters={
            "seed": 42,
            "model_params": {"vocab_size": 50257, "context_length": 1024,
                              "n_layers": 12, "n_heads": 12, "d_model": 768},
            "optimizer_params": {"lr": 3e-4, "weight_decay": 0.01, "betas": [0.9, 0.95]},
            "training": {"total_steps": 100_000, "save_every": 1000, "batch_size": 32},
        },
        dependencies=["Tokenize"],
        resources=ResourcesSpec(gpu_type="A100", gpu_count=4),
    )

RunConfig(
        metadata={"run_id": "gpt2-124m-run3"},
        jobs={
            "Tokenize": JobEntry(parameters={"vocab_size": 50257}),
            "Train": JobEntry(
                parameters={"seed": 42, "training": {"total_steps": 100_000}},
                dependencies=["Tokenize"],
                resources=ResourcesSpec(gpu_type="A100", gpu_count=4),
            ),
        },
    )